In [ ]:
import re
import os
import csv
import json
import pandas as pd

# klines 디렉토리 내의 파일 탐색
klines_file_path = "../data/binance/futures/um/monthly/klines/BTCUSDT/15m"
klines_use_columns = [
    "open_time",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "quote_volume",
    "count",
    "taker_buy_volume",
    "taker_buy_quote_volume",
]
klines_file_list = os.listdir(klines_file_path)
klines_file_list.sort()

# position distribution(pd) 디렉토리 내의 파일 탐색
pd_file_path = "../data/binance/futures/um/monthly/position_distribution/900000"
pd_file_list = os.listdir(pd_file_path)
pd_file_list.sort()

# 저장할 데이터 프레임 생성
result_path = "../data/binance/futures/um/dataset/"
retail_result_df = None
institutional_result_df = None
action_dims = 20

# l_i, s_i column 순서 정렬
column_order = []
for i in range(action_dims):
    column_order.extend([f"l_{i}"])
for i in range(action_dims):
    column_order.extend([f"s_{i}"])

for pd_file in pd_file_list:
    # pd_df 읽기
    pd_df = pd.read_csv(os.path.join(pd_file_path, pd_file), index_col=0)

    # JSON 형태로 저장된 리스트 컬럼 변환
    pd_df["retail_long"] = pd_df["retail_long"].apply(
        lambda x: json.loads(x) if isinstance(x, str) else x
    )
    pd_df["retail_short"] = pd_df["retail_short"].apply(
        lambda x: json.loads(x) if isinstance(x, str) else x
    )
    pd_df["institutional_long"] = pd_df["institutional_long"].apply(
        lambda x: json.loads(x) if isinstance(x, str) else x
    )
    pd_df["institutional_short"] = pd_df["institutional_short"].apply(
        lambda x: json.loads(x) if isinstance(x, str) else x
    )

    # 날짜/시간 열을 인덱스로 설정
    pd_df.index = pd.to_datetime(pd_df.index)

    # retail_pd_df와 institutional_pd_df 생성
    retail_pd_df = pd.DataFrame(index=pd_df.index)
    institutional_pd_df = pd.DataFrame(index=pd_df.index)

    # 리스트 데이터를 개별 컬럼으로 확장
    for i in range(action_dims):
        retail_pd_df[f"l_{i}"] = pd_df["retail_long"].apply(
            lambda x: x[i] if isinstance(x, list) and len(x) > i else None
        )
        retail_pd_df[f"s_{i}"] = pd_df["retail_short"].apply(
            lambda x: x[i] if isinstance(x, list) and len(x) > i else None
        )
        institutional_pd_df[f"l_{i}"] = pd_df["institutional_long"].apply(
            lambda x: x[i] if isinstance(x, list) and len(x) > i else None
        )
        institutional_pd_df[f"s_{i}"] = pd_df["institutional_short"].apply(
            lambda x: x[i] if isinstance(x, list) and len(x) > i else None
        )

    retail_pd_df = retail_pd_df[column_order]
    institutional_pd_df = institutional_pd_df[column_order]

    # pd_df file 이름에서 년도, 월 추출
    match = re.search(r"BTCUSDT-pd-(\d{4})-(\d{2})\.csv", pd_file)
    if not match:
        print(f"파일명 형식이 맞지 않습니다: {pd_file}")
        continue
    year, month = match.groups()

    # 년도, 월에 따라서 klines_file_list에서 해당 파일 찾기
    klines_file = [
        file for file in klines_file_list if file.endswith(f"{year}-{month}.csv")
    ][0]
    klines_df = pd.read_csv(
        os.path.join(klines_file_path, klines_file),
        index_col=0,
        usecols=klines_use_columns,
    )

    # open_time을 datetime으로 변환, 30분 뒤로 밀기
    klines_df.index = pd.to_datetime(klines_df.index, unit="ms")
    klines_df.index = klines_df.index + pd.Timedelta(minutes=30)

    # klines_df와 retail_pd_df, institutional_pd_df 결합
    retail_combined_df = pd.concat([klines_df, retail_pd_df], axis=1)
    institutional_combined_df = pd.concat([klines_df, institutional_pd_df], axis=1)

    if retail_result_df is None:
        retail_result_df = retail_combined_df
    else:
        retail_result_df = pd.concat([retail_result_df, retail_combined_df])

    if institutional_result_df is None:
        institutional_result_df = institutional_combined_df
    else:
        institutional_result_df = pd.concat(
            [institutional_result_df, institutional_combined_df]
        )

# 9:1 비율로 train/test 데이터 분리
ratio = 0.9
retail_train_df = retail_result_df.iloc[: int(len(retail_result_df) * ratio)]
retail_test_df = retail_result_df.iloc[int(len(retail_result_df) * ratio) :]

institutional_train_df = institutional_result_df.iloc[
    : int(len(institutional_result_df) * ratio)
]
institutional_test_df = institutional_result_df.iloc[
    int(len(institutional_result_df) * ratio) :
]

# 디렉토리 생성
os.makedirs(result_path, exist_ok=True)

# 각각의 데이터프레임을 CSV 파일로 저장
retail_train_df.to_csv(
    os.path.join(result_path, "retail_train.csv"),
    index=True,
    quoting=csv.QUOTE_NONNUMERIC,
)
retail_test_df.to_csv(
    os.path.join(result_path, "retail_test.csv"),
    index=True,
    quoting=csv.QUOTE_NONNUMERIC,
)
institutional_train_df.to_csv(
    os.path.join(result_path, "institutional_train.csv"),
    index=True,
    quoting=csv.QUOTE_NONNUMERIC,
)
institutional_test_df.to_csv(
    os.path.join(result_path, "institutional_test.csv"),
    index=True,
    quoting=csv.QUOTE_NONNUMERIC,
)

In [2]:
retail_train_df

,open,high,low,close,volume,quote_volume,count,taker_buy_volume,taker_buy_quote_volume,l_0,...,s_10,s_11,s_12,s_13,s_14,s_15,s_16,s_17,s_18,s_19
2022-06-01 00:30:00,31797.9,31880.0,31680.0,31812.8,7559.249,2.402725e+08,69640,3747.846,1.191518e+08,0.0,...,0.0,0.0,0.000000,0.000000,0.038350,0.207004,0.0,0.0,0.0,0.0
2022-06-01 01:00:00,31812.8,31986.1,31781.8,31925.5,6029.639,1.923678e+08,62822,3147.042,1.004210e+08,0.0,...,0.0,0.0,0.000000,0.000000,0.023761,0.214226,0.0,0.0,0.0,0.0
2022-06-01 01:30:00,31925.5,31938.6,31833.0,31895.4,3072.287,9.796032e+07,36014,1458.408,4.650607e+07,0.0,...,0.0,0.0,0.000000,0.000000,0.040348,0.212199,0.0,0.0,0.0,0.0
2022-06-01 02:00:00,31895.4,31923.0,31825.7,31914.9,2749.653,8.763943e+07,33191,1257.258,4.007458e+07,0.0,...,0.0,0.0,0.000000,0.006528,0.047404,0.191638,0.0,0.0,0.0,0.0
2022-06-01 02:30:00,31914.9,31915.0,31743.6,31772.0,4111.918,1.308276e+08,45232,1804.444,5.740528e+07,0.0,...,0.0,0.0,0.000000,0.035485,0.046936,0.189128,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-18 10:00:00,104250.0,104287.4,104010.6,104015.2,1608.719,1.675509e+08,37201,727.186,7.574464e+07,0.0,...,0.0,0.0,0.076935,0.206603,0.000000,0.000000,0.0,0.0,0.0,0.0
2024-12-18 10:30:00,104015.2,104314.1,103950.0,104250.4,1684.474,1.753970e+08,43314,931.638,9.700446e+07,0.0,...,0.0,0.0,0.064480,0.156257,0.000000,0.000000,0.0,0.0,0.0,0.0
2024-12-18 11:00:00,104250.1,104719.4,104163.8,104499.1,3603.146,3.764366e+08,58957,2295.367,2.398467e+08,0.0,...,0.0,0.0,0.000000,0.070688,0.146095,0.000000,0.0,0.0,0.0,0.0
2024-12-18 11:30:00,104499.1,104763.2,104455.1,104698.3,2292.677,2.399072e+08,48521,1132.457,1.185079e+08,0.0,...,0.0,0.0,0.083966,0.153385,0.000000,0.000000,0.0,0.0,0.0,0.0
